In [6]:
import pandas as pd
import numpy as np
import torch
import spacy

In [7]:
df = pd.read_parquet(
    "../data/processed/crisisbench_train_processed.parquet"
)

In [8]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_path = "../models/distilbert-crisisbench"

tokenizer = AutoTokenizer.from_pretrained(model_path)

model = AutoModelForSequenceClassification.from_pretrained(
    model_path
)

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 5760.57it/s]


In [11]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model.to(device)
model.eval()

print("Using:", device)

Using: cuda


In [12]:
nlp = spacy.load("en_core_web_sm")

In [13]:
severity_keywords = {
    "death", "dead", "killed", "fatality", "casualties",
    "explosion", "attack", "collapse", "destroyed",
    "missing", "injured", "victim", "disaster"
}

urgency_keywords = {
    "urgent", "emergency", "help", "rescue",
    "trapped", "immediate", "evacuate", "evacuation",
    "need", "missing", "danger", "warning"
}

In [14]:
def keyword_risk_score(text):
    words = set(text.lower().split())

    severity_matches = words.intersection(severity_keywords)
    urgency_matches = words.intersection(urgency_keywords)

    return {
        "severity_count": len(severity_matches),
        "urgency_count": len(urgency_matches),
        "severity_terms": list(severity_matches),
        "urgency_terms": list(urgency_matches)
    }

In [15]:
keyword_risk_score(
    "Explosion killed victims and people urgently need rescue help."
)

{'severity_count': 2,
 'urgency_count': 2,
 'severity_terms': ['killed', 'explosion'],
 'urgency_terms': ['rescue', 'need']}

In [16]:
def entity_risk_score(text):
    doc = nlp(text)

    entities = [
        {
            "text": ent.text,
            "label": ent.label_
        }
        for ent in doc.ents
    ]

    return {
        "entity_count": len(entities),
        "entities": entities
    }

In [17]:
entity_risk_score(
    "A major earthquake has hit Kathmandu, Nepal and rescue teams are helping victims."
)

{'entity_count': 2,
 'entities': [{'text': 'Kathmandu', 'label': 'GPE'},
  {'text': 'Nepal', 'label': 'GPE'}]}

In [18]:
def calculate_risk(text):
    text_lower = text.lower()

    # 1. Informativeness
    prediction = predict_post(text)

    score = 40 if prediction["prediction"] == "informative" else 0

    # 2. Keyword signals
    keyword_scores = keyword_risk_score(text_lower)

    severity_score = min(
        keyword_scores["severity_count"] * 15,
        30
    )

    urgency_score = min(
        keyword_scores["urgency_count"] * 10,
        20
    )

    # 3. Entity signal
    entity_scores = entity_risk_score(text)

    entity_score = min(
        entity_scores["entity_count"] * 5,
        10
    )

    # Final score
    total_score = min(
        score + severity_score + urgency_score + entity_score,
        100
    )

    return {
        "risk_score": total_score,
        "prediction": prediction["prediction"],
        "confidence": prediction["confidence"],
        "severity_terms": keyword_scores["severity_terms"],
        "urgency_terms": keyword_scores["urgency_terms"],
        "entities": entity_scores["entities"]
    }

In [22]:
def predict_post(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        outputs = model(**inputs)

    probabilities = torch.softmax(outputs.logits, dim=-1)

    predicted_class = probabilities.argmax().item()
    confidence = probabilities[0, predicted_class].item()

    return {
        "prediction": model.config.id2label[predicted_class],
        "confidence": round(confidence, 4)
    }

In [23]:
def keyword_risk_score(text):
    words = set(text.lower().split())

    severity_matches = words.intersection(severity_keywords)
    urgency_matches = words.intersection(urgency_keywords)

    return {
        "severity_count": len(severity_matches),
        "urgency_count": len(urgency_matches),
        "severity_terms": list(severity_matches),
        "urgency_terms": list(urgency_matches)
    }

In [24]:
def entity_risk_score(text):
    doc = nlp(text)

    entities = [
        {
            "text": ent.text,
            "label": ent.label_
        }
        for ent in doc.ents
    ]

    return {
        "entity_count": len(entities),
        "entities": entities
    }

In [25]:
calculate_risk(
    "Major explosion killed several people in Kathmandu. Emergency rescue teams urgently need help."
)

{'risk_score': 95,
 'prediction': 'informative',
 'confidence': 0.9925,
 'severity_terms': ['killed', 'explosion'],
 'urgency_terms': ['rescue', 'emergency', 'need'],
 'entities': [{'text': 'Kathmandu', 'label': 'GPE'}]}

In [26]:
examples = [
    "Heavy rainfall is expected tomorrow.",
    
    "URGENT! Explosion reported near the city. People are injured and need immediate rescue.",
    
    "Had a great dinner with my friends today!"
]

for text in examples:
    result = calculate_risk(text)
    print("TEXT:", text)
    print("RESULT:", result)
    print("-" * 100)

TEXT: Heavy rainfall is expected tomorrow.
RESULT: {'risk_score': 45, 'prediction': 'informative', 'confidence': 0.8087, 'severity_terms': [], 'urgency_terms': [], 'entities': [{'text': 'tomorrow', 'label': 'DATE'}]}
----------------------------------------------------------------------------------------------------
TEXT: URGENT! Explosion reported near the city. People are injured and need immediate rescue.
RESULT: {'risk_score': 90, 'prediction': 'informative', 'confidence': 0.9948, 'severity_terms': ['injured', 'explosion'], 'urgency_terms': ['need', 'immediate'], 'entities': []}
----------------------------------------------------------------------------------------------------
TEXT: Had a great dinner with my friends today!
RESULT: {'risk_score': 5, 'prediction': 'not_informative', 'confidence': 0.9891, 'severity_terms': [], 'urgency_terms': [], 'entities': [{'text': 'today', 'label': 'DATE'}]}
---------------------------------------------------------------------------------------

In [27]:
def get_risk_level(score):
    if score >= 75:
        return "CRITICAL"
    elif score >= 50:
        return "HIGH"
    elif score >= 25:
        return "MEDIUM"
    else:
        return "LOW"

In [28]:
def calculate_risk(text):
    prediction = predict_post(text)

    score = 40 if prediction["prediction"] == "informative" else 0

    keyword_scores = keyword_risk_score(text)

    severity_score = min(
        keyword_scores["severity_count"] * 15,
        30
    )

    urgency_score = min(
        keyword_scores["urgency_count"] * 10,
        20
    )

    entity_scores = entity_risk_score(text)

    entity_score = min(
        entity_scores["entity_count"] * 5,
        10
    )

    total_score = min(
        score + severity_score + urgency_score + entity_score,
        100
    )

    return {
        "risk_score": total_score,
        "risk_level": get_risk_level(total_score),
        "prediction": prediction["prediction"],
        "confidence": prediction["confidence"],
        "severity_terms": keyword_scores["severity_terms"],
        "urgency_terms": keyword_scores["urgency_terms"],
        "entities": entity_scores["entities"]
    }

In [29]:
calculate_risk(
    "URGENT! Explosion reported near Kathmandu. Several people are injured and trapped. Rescue teams need immediate help."
)

{'risk_score': 95,
 'risk_level': 'CRITICAL',
 'prediction': 'informative',
 'confidence': 0.9923,
 'severity_terms': ['injured', 'explosion'],
 'urgency_terms': ['rescue', 'need', 'immediate'],
 'entities': [{'text': 'Kathmandu', 'label': 'GPE'}]}

In [30]:
def calculate_risk(text):
    prediction = predict_post(text)

    # Risk starts from severity + urgency, not informativeness
    keyword_scores = keyword_risk_score(text)

    severity_score = min(
        keyword_scores["severity_count"] * 20,
        40
    )

    urgency_score = min(
        keyword_scores["urgency_count"] * 15,
        30
    )

    # Entity presence provides contextual evidence
    entity_scores = entity_risk_score(text)

    entity_score = min(
        entity_scores["entity_count"] * 5,
        15
    )

    # Small priority contribution from informativeness
    informative_bonus = 10 if prediction["prediction"] == "informative" else 0

    total_score = min(
        severity_score +
        urgency_score +
        entity_score +
        informative_bonus,
        100
    )

    return {
        "risk_score": total_score,
        "risk_level": get_risk_level(total_score),
        "prediction": prediction["prediction"],
        "confidence": prediction["confidence"],
        "severity_terms": keyword_scores["severity_terms"],
        "urgency_terms": keyword_scores["urgency_terms"],
        "entities": entity_scores["entities"]
    }

In [31]:
examples = [
    "Heavy rainfall is expected tomorrow.",
    
    "URGENT! Explosion reported near the city. People are injured and need immediate rescue.",
    
    "Had a great dinner with my friends today!"
]

for text in examples:
    result = calculate_risk(text)
    print("TEXT:", text)
    print("RISK SCORE:", result["risk_score"])
    print("RISK LEVEL:", result["risk_level"])
    print("-" * 80)

TEXT: Heavy rainfall is expected tomorrow.
RISK SCORE: 15
RISK LEVEL: LOW
--------------------------------------------------------------------------------
TEXT: URGENT! Explosion reported near the city. People are injured and need immediate rescue.
RISK SCORE: 80
RISK LEVEL: CRITICAL
--------------------------------------------------------------------------------
TEXT: Had a great dinner with my friends today!
RISK SCORE: 5
RISK LEVEL: LOW
--------------------------------------------------------------------------------


In [32]:
sample_posts = df["text"].sample(10, random_state=42)

for text in sample_posts:
    result = calculate_risk(text)

    print("TEXT:", text)
    print("RISK:", result["risk_score"])
    print("LEVEL:", result["risk_level"])
    print("PREDICTION:", result["prediction"])
    print("-" * 100)

TEXT: Slowly moving 'Ruby' to bring torrential rains http://t.co/1JXuR4APuP #RubyPH
RISK: 0
LEVEL: LOW
PREDICTION: not_informative
----------------------------------------------------------------------------------------------------
TEXT: MT @dineshakula Med supplies required in Bir Hospital. Out of medical supplies http://t.co/4pPhg2aVhg #Kathmandu #NepalQuake #hmrd
RISK: 20
LEVEL: LOW
PREDICTION: informative
----------------------------------------------------------------------------------------------------
TEXT: The Chinese Embassy in Manila express condolences to Typhoon Yolanda victims. |via Hua Zhang, embassy spokesman
RISK: 25
LEVEL: MEDIUM
PREDICTION: informative
----------------------------------------------------------------------------------------------------
TEXT: Rwanda remembers genocide victims 20 years on - http://t.co/7ewhwgpybO
RISK: 20
LEVEL: LOW
PREDICTION: informative
---------------------------------------------------------------------------------------------------

In [33]:
def calculate_risk(text):
    prediction = predict_post(text)

    keyword_scores = keyword_risk_score(text)

    severity_score = min(
        keyword_scores["severity_count"] * 20,
        40
    )

    urgency_score = min(
        keyword_scores["urgency_count"] * 15,
        30
    )

    entity_scores = entity_risk_score(text)

    entity_score = min(
        entity_scores["entity_count"] * 5,
        15
    )

    informative_bonus = (
        10 if prediction["prediction"] == "informative" else 0
    )

    total_score = min(
        severity_score +
        urgency_score +
        entity_score +
        informative_bonus,
        100
    )

    return {
        "risk_score": total_score,
        "risk_level": get_risk_level(total_score),
        "prediction": prediction["prediction"],
        "confidence": prediction["confidence"],
        "score_breakdown": {
            "severity": severity_score,
            "urgency": urgency_score,
            "entity_context": entity_score,
            "informative_priority": informative_bonus
        },
        "severity_terms": keyword_scores["severity_terms"],
        "urgency_terms": keyword_scores["urgency_terms"],
        "entities": entity_scores["entities"]
    }

In [34]:
result = calculate_risk(
    "URGENT! Explosion reported near Kathmandu. Several people are injured and trapped. Rescue teams need immediate help."
)

result

{'risk_score': 85,
 'risk_level': 'CRITICAL',
 'prediction': 'informative',
 'confidence': 0.9923,
 'score_breakdown': {'severity': 40,
  'urgency': 30,
  'entity_context': 5,
  'informative_priority': 10},
 'severity_terms': ['injured', 'explosion'],
 'urgency_terms': ['rescue', 'need', 'immediate'],
 'entities': [{'text': 'Kathmandu', 'label': 'GPE'}]}

In [35]:
def generate_alert(risk_result):
    level = risk_result["risk_level"]

    if level == "CRITICAL":
        action = "IMMEDIATE_REVIEW"
        alert = True
    elif level == "HIGH":
        action = "REVIEW"
        alert = True
    elif level == "MEDIUM":
        action = "MONITOR"
        alert = False
    else:
        action = "NO_ACTION"
        alert = False

    return {
        "alert": alert,
        "action": action,
        "risk_level": level,
        "risk_score": risk_result["risk_score"],
        "confidence": risk_result["confidence"],
        "reasons": {
            "severity_terms": risk_result["severity_terms"],
            "urgency_terms": risk_result["urgency_terms"],
            "entities": risk_result["entities"]
        }
    }

In [36]:
def analyze_risk(text):
    risk_result = calculate_risk(text)
    alert_result = generate_alert(risk_result)

    return alert_result

In [37]:
analyze_risk(
    "URGENT! Explosion reported near Kathmandu. Several people are injured and trapped. Rescue teams need immediate help."
)

{'alert': True,
 'action': 'IMMEDIATE_REVIEW',
 'risk_level': 'CRITICAL',
 'risk_score': 85,
 'confidence': 0.9923,
 'reasons': {'severity_terms': ['injured', 'explosion'],
  'urgency_terms': ['rescue', 'need', 'immediate'],
  'entities': [{'text': 'Kathmandu', 'label': 'GPE'}]}}

In [38]:
risk_samples = df[["text", "class_label"]].sample(
    100,
    random_state=42
)

In [39]:
risk_results = []

for text in risk_samples["text"]:
    result = calculate_risk(text)

    risk_results.append({
        "text": text,
        "risk_score": result["risk_score"],
        "risk_level": result["risk_level"],
        "prediction": result["prediction"],
        "confidence": result["confidence"]
    })

In [40]:
risk_df = pd.DataFrame(risk_results)

In [41]:
risk_df["risk_level"].value_counts()

risk_level
LOW         69
MEDIUM      27
HIGH         3
CRITICAL     1
Name: count, dtype: int64

In [42]:
risk_df.sort_values(
    "risk_score",
    ascending=False
)[[
    "text",
    "risk_score",
    "risk_level",
    "prediction",
    "confidence"
]].head(10)

,text,risk_score,risk_level,prediction,confidence
97,"Over 10 days since disasters struck #lka, the ...",80,CRITICAL,informative,0.9898
26,285 dead in Bangladesh collapse: The death tol...,65,HIGH,informative,0.9926
93,"70 killed, hundreds injured by an explosion at...",65,HIGH,informative,0.9934
14,RT @gabbietatad: #reliefPH RT SAN ANTONIO stil...,55,HIGH,informative,0.9922
76,"""US drone strike believed to have killed provi...",45,MEDIUM,informative,0.9832
62,RT @greenhousenyt: My Story on Fertilizer Plan...,45,MEDIUM,informative,0.9938
60,#Puerto #Rico in need of '#Unprecedented $#21b...,40,MEDIUM,informative,0.9370
98,"Yet the Burmese military junta, fearing that c...",40,MEDIUM,informative,0.8048
9,@gomezisourangel: Explosion in Texas? WHAT? I ...,40,MEDIUM,informative,0.9919
5,RT @doh_philippines: Dead bodies do not pose a...,35,MEDIUM,informative,0.9875


In [43]:
def calculate_risk(text):
    prediction = predict_post(text)

    keyword_scores = keyword_risk_score(text)

    severity_score = min(
        keyword_scores["severity_count"] * 20,
        40
    )

    urgency_score = min(
        keyword_scores["urgency_count"] * 15,
        30
    )

    entity_scores = entity_risk_score(text)

    entity_score = min(
        entity_scores["entity_count"] * 5,
        15
    )

    # Confidence-aware informativeness bonus
    if prediction["prediction"] == "informative":
        informative_bonus = round(
            prediction["confidence"] * 10,
            2
        )
    else:
        informative_bonus = 0

    total_score = min(
        severity_score +
        urgency_score +
        entity_score +
        informative_bonus,
        100
    )

    return {
        "risk_score": total_score,
        "risk_level": get_risk_level(total_score),
        "prediction": prediction["prediction"],
        "confidence": prediction["confidence"],
        "score_breakdown": {
            "severity": severity_score,
            "urgency": urgency_score,
            "entity_context": entity_score,
            "informative_priority": informative_bonus
        },
        "severity_terms": keyword_scores["severity_terms"],
        "urgency_terms": keyword_scores["urgency_terms"],
        "entities": entity_scores["entities"]
    }

In [44]:
result = calculate_risk(
    "URGENT! Explosion reported near Kathmandu. Several people are injured and trapped. Rescue teams need immediate help."
)

result

{'risk_score': 84.92,
 'risk_level': 'CRITICAL',
 'prediction': 'informative',
 'confidence': 0.9923,
 'score_breakdown': {'severity': 40,
  'urgency': 30,
  'entity_context': 5,
  'informative_priority': 9.92},
 'severity_terms': ['injured', 'explosion'],
 'urgency_terms': ['rescue', 'need', 'immediate'],
 'entities': [{'text': 'Kathmandu', 'label': 'GPE'}]}

In [45]:
risk_results = []

for text in risk_samples["text"]:
    result = calculate_risk(text)

    risk_results.append({
        "text": text,
        "risk_score": result["risk_score"],
        "risk_level": result["risk_level"],
        "prediction": result["prediction"],
        "confidence": result["confidence"]
    })

risk_df = pd.DataFrame(risk_results)

In [46]:
risk_df["risk_level"].value_counts()

risk_level
LOW         87
MEDIUM       9
HIGH         3
CRITICAL     1
Name: count, dtype: int64

In [47]:
risk_df["risk_score"].describe()

count    100.000000
mean      17.955000
std       14.499485
min        0.000000
25%        9.635000
50%       15.000000
75%       24.377500
max       79.900000
Name: risk_score, dtype: float64

In [48]:
import json

risk_config = {
    "severity_weight": 20,
    "severity_cap": 40,
    "urgency_weight": 15,
    "urgency_cap": 30,
    "entity_weight": 5,
    "entity_cap": 15,
    "informative_bonus_max": 10,
    "risk_levels": {
        "LOW": [0, 24],
        "MEDIUM": [25, 49],
        "HIGH": [50, 74],
        "CRITICAL": [75, 100]
    }
}

with open("../models/risk_config.json", "w") as f:
    json.dump(risk_config, f, indent=4)

In [49]:
import os

print(os.path.exists("../models/risk_config.json"))

True
